# Интерактивный разбор растеризации 3D Gaussian Splatting

Этот ноутбук декомпозирует алгоритм из `3dgs.ipynb` на небольшие
последовательные стадии. Каждая code-ячейка работает с реальными данными
сцены `bonsai`, печатает формы тензоров и показывает небольшие срезы
фактических значений.

Запускайте ячейки строго сверху вниз: каждая стадия использует тензоры,
подготовленные предыдущей стадией.

Полный путь данных:

```text
load parameters
→ build world covariance
→ project centers
→ frustum filter
→ project covariance to 2D
→ stabilize eigenvalues
→ sort by depth
→ build screen-space AABBs
→ enumerate Gaussian/tile intersections
→ stably group intersections by tile
→ evaluate Gaussians per tile
→ front-to-back alpha compositing
→ final image
```

Текущая реализация проверена на Apple Silicon с backend `mps`. Если MPS
недоступен, ноутбук автоматически попробует CUDA, а затем CPU.


## 0. Импорты, устройство и параметры

Здесь задаются те же основные параметры, что используются в основном
ноутбуке: тайл `16×16`, camera ID 10 и половинное разрешение.

`show_tensor` всегда переносит на CPU только небольшой переданный срез,
поэтому диагностическая печать не копирует целиком многомиллионные
тензоры.


In [ ]:
from pathlib import Path
from time import perf_counter

import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
import torch

from util import (
    build_covariance,
    eigh_2x2,
    inv2x2,
    load_cameras,
    project_points,
    scale_intrinsics,
)

torch.set_printoptions(precision=5, sci_mode=False, linewidth=120)

if torch.backends.mps.is_available():
    device = torch.device("mps")
elif torch.cuda.is_available():
    device = torch.device("cuda")
else:
    device = torch.device("cpu")

scene = "bonsai"
CAM_ID = 10
T = 16
near = 2e-3
far = 100.0
pix_guard = 64
min_conis = 1e-6
chi_square_clip = 9.21
alpha_max = 0.99
alpha_cutoff = 1 / 255.0

def show_tensor(name, tensor, max_items=6):
    flat = tensor.detach().reshape(-1)
    sample = flat[:max_items].cpu()
    print(f"{name}: shape={tuple(tensor.shape)}, dtype={tensor.dtype}, device={tensor.device}")
    print(f"  first {sample.numel()} values: {sample}")

def synchronize():
    if device.type == "mps":
        torch.mps.synchronize()
    elif device.type == "cuda":
        torch.cuda.synchronize()

print("device:", device)
print("tile size:", T)


## 1. Загрузка реальных параметров сцены

Для каждого гауссиана загружаются:

- центр `[x, y, z]`;
- raw opacity;
- цвет;
- логарифмические масштабы;
- кватернион `[x, y, z, w]`.

Мировая ковариация строится как:

```text
scale = exp(scale_raw)
Sigma_world = R * S * S * transpose(R)
```

В конце ячейки выводится настоящий первый гауссиан: его позиция,
активированные масштабы, длина кватерниона и матрица ковариации.

> **Отличие от CUDA-референса:** референс вызывает
> `build_sigma_from_params`; эта версия использует `build_covariance`
> из `util.py` с той же формулой `R S² Rᵀ`.


In [ ]:
def load_parameter(path):
    value = torch.load(path, weights_only=False)
    if isinstance(value, np.ndarray):
        value = torch.from_numpy(value)
    return value.to(device)

pos = load_parameter("out_bonsai/pos_param.pt")
opacity_raw = load_parameter("out_bonsai/alpha_raw_param.pt")
f_dc = load_parameter("out_bonsai/f_dc.pt")
scale_raw = load_parameter("out_bonsai/scale_raw.pt")
rot_raw = load_parameter("out_bonsai/rot_raw.pt")

color = torch.sigmoid(0.282 * f_dc)
sigma = build_covariance(scale_raw, rot_raw)
dtype = pos.dtype

cam_parameters = np.load(
    f"out_colmap/{scene}/cam_meta.npy",
    allow_pickle=True,
).item()

H_source = cam_parameters["height"]
W_source = cam_parameters["width"]
fx_source = cam_parameters["fx"]
fy_source = cam_parameters["fy"]
cx_source = W_source / 2
cy_source = H_source / 2

H = H_source // 2
W = W_source // 2
fx, fy, cx, cy = scale_intrinsics(
    H,
    W,
    H_source,
    W_source,
    fx_source,
    fy_source,
    cx_source,
    cy_source,
)

c2ws, image_paths = load_cameras(
    f"out_colmap/{scene}/cameras.npy",
    f"image_data/{scene}/images_2",
)
c2w = c2ws[CAM_ID].to(device)
image_path = image_paths[CAM_ID]

print(f"Gaussians: {pos.shape[0]:,}")
print(f"source image: {W_source} × {H_source}")
print(f"render image: {W} × {H}")
print(f"intrinsics: fx={fx:.3f}, fy={fy:.3f}, cx={cx:.3f}, cy={cy:.3f}")
show_tensor("pos[:3]", pos[:3])
show_tensor("exp(scale_raw[:3])", torch.exp(scale_raw[:3]))
show_tensor("quaternion norms[:3]", torch.linalg.norm(rot_raw[:3], dim=-1))
print("sigma[0] =\n", sigma[0].detach().cpu())

del f_dc, scale_raw, rot_raw


## 2. Проекция центров и frustum-фильтрация

`project_points` преобразует мировые координаты в координаты камеры
`(x, y, z)`, а затем вычисляет пиксельные координаты:

```text
u = fx*x/z + cx
v = fy*y/z + cy
```

После проекции отбрасываются центры за диапазоном `near..far` и далеко
за границами изображения. `pix_guard` оставляет небольшой запас, потому
что центр может быть вне кадра, а эллипс гауссиана всё ещё пересекать
изображение.


In [ ]:
uv, x, y, z = project_points(pos, c2w, H, W, fx, fy, cx, cy)

u = uv[:, 0]
v = uv[:, 1]
frustum = (
    (u > -pix_guard)
    & (u < W + pix_guard)
    & (v > -pix_guard)
    & (v < H + pix_guard)
    & (z > near)
    & (z < far)
)

input_count = pos.shape[0]
frustum_count = int(frustum.sum().item())
first_visible_input_ids = torch.nonzero(frustum, as_tuple=False).flatten()[:3]

print(f"input Gaussians: {input_count:,}")
print(f"after frustum filter: {frustum_count:,}")
print("first visible input IDs:", first_visible_input_ids.cpu().tolist())
show_tensor("uv[first visible]", uv[first_visible_input_ids])
show_tensor("z[first visible]", z[first_visible_input_ids])

uv = uv[frustum]
pos = pos[frustum]
color = color[frustum]
opacity = torch.sigmoid(opacity_raw[frustum]).squeeze(-1).clamp(0, 0.999)
x = x[frustum]
y = y[frustum]
z = z[frustum]
sigma = sigma[frustum]

show_tensor("activated opacity[:6]", opacity[:6])
del opacity_raw, frustum


## 3. Проекция ковариации 3×3 в экранную ковариацию 2×2

Сначала мировая ковариация поворачивается в координаты камеры:

```text
Sigma_camera = R_wc * Sigma_world * transpose(R_wc)
```

Затем применяется Якоби перспективной проекции:

```text
J = [[fx/z,    0, -fx*x/z²],
     [   0, fy/z, -fy*y/z²]]

Sigma_uv = J * Sigma_camera * transpose(J)
```

В конце ячейки печатаются реальные `J`, `Sigma_camera` и `Sigma_uv`
первого оставшегося гауссиана.


In [ ]:
Rcw = c2w[:3, :3]
Rwc = Rcw.T

J = torch.zeros((pos.shape[0], 2, 3), device=device, dtype=dtype)
J[:, 0, 0] = fx / z
J[:, 1, 1] = fy / z
J[:, 0, 2] = -fx * x / (z * z)
J[:, 1, 2] = -fy * y / (z * z)

sigma_camera = Rwc.unsqueeze(0) @ sigma @ Rwc.T.unsqueeze(0)
sigma_uv = J @ sigma_camera @ J.transpose(1, 2)
sigma_uv = 0.5 * (sigma_uv + sigma_uv.transpose(1, 2))

print("J[0] =\n", J[0].detach().cpu())
print("Sigma_camera[0] =\n", sigma_camera[0].detach().cpu())
print("Sigma_uv[0] =\n", sigma_uv[0].detach().cpu())

del pos, sigma, J, sigma_camera, x, y
if device.type == "mps":
    torch.mps.empty_cache()


## 4. Собственные значения и стабилизация эллипсов

Собственные значения `Sigma_uv` — дисперсии вдоль малой и большой осей
экранного эллипса. Они ограничиваются диапазоном `1e-6..1e4`, после чего
матрица собирается обратно.

> **Отличие от CUDA-референса:** автор использует
> `torch.linalg.eigh`. Для MPS здесь применяется аналитическая
> `eigh_2x2`, чтобы не переносить матрицы на CPU.


In [ ]:
evals_before, evecs = eigh_2x2(sigma_uv)
evals = torch.clamp(evals_before, min=1e-6, max=1e4)
sigma_uv = evecs @ torch.diag_embed(evals) @ evecs.transpose(1, 2)

finite = torch.isfinite(sigma_uv.reshape(sigma_uv.shape[0], -1)).all(dim=-1)
finite_count = int(finite.sum().item())

print("eigenvalues before clamp, first 5:\n", evals_before[:5].detach().cpu())
print("eigenvalues after clamp, first 5:\n", evals[:5].detach().cpu())
print(f"finite covariance matrices: {finite_count:,} / {sigma_uv.shape[0]:,}")
print("reconstructed Sigma_uv[0] =\n", sigma_uv[0].detach().cpu())

uv = uv[finite]
color = color[finite]
opacity = opacity[finite]
z = z[finite]
sigma_uv = sigma_uv[finite]
evals = evals[finite]

del evals_before, evecs, finite


## 5. Глобальная сортировка по глубине

Front-to-back compositing требует порядка от ближних гауссианов к
дальним. После сортировки индекс в массивах становится одновременно
позицией в глобальном порядке глубины. Это свойство позже позволит
выполнить стабильную сортировку только по `flat_tile_id`.


In [ ]:
z, depth_order = torch.sort(z, descending=False)
uv = uv[depth_order]
color = color[depth_order]
opacity = opacity[depth_order]
sigma_uv = sigma_uv[depth_order]
evals = evals[depth_order]

u = uv[:, 0]
v = uv[:, 1]

assert bool(torch.all(z[1:] >= z[:-1]).item())
print("first 10 sorted depths:\n", z[:10].detach().cpu())
print("last 3 sorted depths:\n", z[-3:].detach().cpu())
del depth_order, z, uv


## 6. Экранный AABB и диапазоны тайлов

Радиус круга, содержащего экранный эллипс, оценивается по наибольшему
собственному значению:

```text
radius = ceil(3 * sqrt(major_variance))
```

Пиксельный AABB переводится в целочисленные диапазоны `tile_u` и
`tile_v`. Декартово произведение этих диапазонов и есть
**прямоугольник тайлов одного гауссиана**.

Например, `tile_u=3..4` и `tile_v=1..3` описывают 6 комбинаций:
`(3,1)`, `(3,2)`, `(3,3)`, `(4,1)`, `(4,2)`, `(4,3)`.

В этой ячейке пример выбирается не вручную: значения печатаются для
настоящего гауссиана текущей сцены.


In [ ]:
major_variance = evals[:, 1].clamp_min(1e-12).clamp_max(1e4)
radius = torch.ceil(3.0 * torch.sqrt(major_variance)).to(torch.int64)

umin = torch.floor(u - radius).to(torch.int64)
umax = torch.floor(u + radius).to(torch.int64)
vmin = torch.floor(v - radius).to(torch.int64)
vmax = torch.floor(v + radius).to(torch.int64)

on_screen = (umax >= 0) & (umin < W) & (vmax >= 0) & (vmin < H)
if not bool(on_screen.any().item()):
    raise RuntimeError("There are no Gaussians on screen")

u = u[on_screen]
v = v[on_screen]
color = color[on_screen]
opacity = opacity[on_screen]
sigma_uv = sigma_uv[on_screen]
umin = umin[on_screen].clamp(0, W - 1)
umax = umax[on_screen].clamp(0, W - 1)
vmin = vmin[on_screen].clamp(0, H - 1)
vmax = vmax[on_screen].clamp(0, H - 1)

umin_tile = (umin // T).to(torch.int64)
umax_tile = (umax // T).to(torch.int64)
vmin_tile = (vmin // T).to(torch.int64)
vmax_tile = (vmax // T).to(torch.int64)

n_u = umax_tile - umin_tile + 1
n_v = vmax_tile - vmin_tile + 1
num_tiles_per_gaussian = n_u * n_v

num_gaussians = umin_tile.shape[0]
num_tile_intersections = int(num_tiles_per_gaussian.sum().item())

candidates = torch.nonzero(
    (num_tiles_per_gaussian >= 4) & (num_tiles_per_gaussian <= 16),
    as_tuple=False,
).flatten()
sample_gaussian_id = int(candidates[0].item()) if candidates.numel() else 0

print(f"on-screen Gaussians: {num_gaussians:,}")
print(f"Gaussian/tile intersections K: {num_tile_intersections:,}")
print("sample Gaussian ID:", sample_gaussian_id)
print(
    "pixel AABB:",
    (int(umin[sample_gaussian_id]), int(vmin[sample_gaussian_id])),
    "..",
    (int(umax[sample_gaussian_id]), int(vmax[sample_gaussian_id])),
)
print(
    "tile ranges:",
    f"u={int(umin_tile[sample_gaussian_id])}..{int(umax_tile[sample_gaussian_id])},",
    f"v={int(vmin_tile[sample_gaussian_id])}..{int(vmax_tile[sample_gaussian_id])}",
)
print(
    "n_u, n_v, n_u*n_v:",
    int(n_u[sample_gaussian_id]),
    int(n_v[sample_gaussian_id]),
    int(num_tiles_per_gaussian[sample_gaussian_id]),
)
del evals, major_variance, radius, on_screen


## 7. Компактный список пересечений «гауссиан × тайл»

Вместо плотного тензора `[Ns, max_u, max_v]` создаются только `K`
существующих пересечений.

`gaussian_ids` повторяет каждый ID нужное количество раз. `cumsum`
вычисляет начало сегмента каждого гауссиана, а `local_tile_ids`
сбрасывается в ноль в начале каждого сегмента:

```text
counts       = [3, 2]
gaussian_ids = [0, 0, 0, 1, 1]
starts       = [0, 3]
local IDs    = [0, 1, 2, 0, 1]
```

Деление на `n_v` даёт локальную координату `u`, остаток — локальную
координату `v`. Поэтому `v` изменяется быстрее.

> **Отличие от CUDA-референса:** референс сначала создаёт плотные
> `tile_u`, `tile_v` и `mask`. Компактный вариант хранит ровно `K`
> элементов и снижает пиковое потребление памяти.


In [ ]:
gaussian_ids = torch.repeat_interleave(
    torch.arange(num_gaussians, device=device, dtype=torch.int64),
    num_tiles_per_gaussian,
    output_size=num_tile_intersections,
)

starts_per_gaussian = torch.cumsum(num_tiles_per_gaussian, dim=0)
starts_per_gaussian = starts_per_gaussian - num_tiles_per_gaussian

local_tile_ids = (
    torch.arange(num_tile_intersections, device=device, dtype=torch.int64)
    - starts_per_gaussian[gaussian_ids]
)
local_tile_u = local_tile_ids // n_v[gaussian_ids]
local_tile_v = local_tile_ids % n_v[gaussian_ids]

flat_tile_u = umin_tile[gaussian_ids] + local_tile_u
flat_tile_v = vmin_tile[gaussian_ids] + local_tile_v

num_tiles_u = (W + T - 1) // T
num_tiles_v = (H + T - 1) // T
flat_tile_id = flat_tile_v * num_tiles_u + flat_tile_u

sample_mask = gaussian_ids == sample_gaussian_id
sample_positions = torch.nonzero(sample_mask, as_tuple=False).flatten()
sample_positions = sample_positions[:16]

print(f"tile grid: {num_tiles_u} × {num_tiles_v}")
print("sample local IDs:", local_tile_ids[sample_positions].cpu().tolist())
print("sample local u:", local_tile_u[sample_positions].cpu().tolist())
print("sample local v:", local_tile_v[sample_positions].cpu().tolist())
print("sample global tile u:", flat_tile_u[sample_positions].cpu().tolist())
print("sample global tile v:", flat_tile_v[sample_positions].cpu().tolist())
print("sample flat tile IDs:", flat_tile_id[sample_positions].cpu().tolist())
del sample_mask, sample_positions


## 8. Стабильная сортировка и диапазоны непустых тайлов

До binning гауссианы уже отсортированы по глубине, а исходный
`gaussian_ids` не убывает. Поэтому достаточно стабильно отсортировать
пары только по `flat_tile_id`:

```text
stable sort by flat_tile_id
→ equal tile IDs become consecutive
→ original near-to-far order remains inside every tile
```

`unique_consecutive` после сортировки возвращает непустые тайлы и число
гауссианов в каждом из них. Префиксная сумма создаёт диапазоны
`[start, end)`.

> **Отличие от CUDA-референса:** референс сортирует большой составной
> `int64`-ключ `tile_id * M + depth_order`. На MPS это создавало
> тайловые артефакты. Стабильная сортировка непосредственно по
> `flat_tile_id` устраняет большие ключи и сохраняет глубинный порядок.


In [ ]:
tile_ids_1d, permutation = torch.sort(flat_tile_id, stable=True)
gaussian_ids = gaussian_ids[permutation]

unique_tile_ids, gaussians_per_tile = torch.unique_consecutive(
    tile_ids_1d,
    return_counts=True,
)
start = torch.zeros_like(unique_tile_ids)
start[1:] = torch.cumsum(gaussians_per_tile[:-1], dim=0)
end = start + gaussians_per_tile

tile_order_ok = bool(torch.all(tile_ids_1d[1:] >= tile_ids_1d[:-1]).item())
same_tile = tile_ids_1d[1:] == tile_ids_1d[:-1]
depth_order_ok = bool(
    torch.all(gaussian_ids[1:][same_tile] >= gaussian_ids[:-1][same_tile]).item()
)

center_tile_id = (H // 2 // T) * num_tiles_u + (W // 2 // T)
center_distance = torch.abs(unique_tile_ids - center_tile_id)
inspection_group_index = int(torch.argmin(center_distance).item())

print(f"non-empty tiles U: {unique_tile_ids.shape[0]:,}")
print("first 10 tile IDs:", unique_tile_ids[:10].cpu().tolist())
print("first 10 Gaussian counts:", gaussians_per_tile[:10].cpu().tolist())
print("tile IDs are sorted:", tile_order_ok)
print("depth order inside tiles is preserved:", depth_order_ok)
print("inspection tile ID:", int(unique_tile_ids[inspection_group_index].item()))
print("Gaussians in inspection tile:", int(gaussians_per_tile[inspection_group_index].item()))

del (
    permutation,
    tile_ids_1d,
    same_tile,
    center_distance,
    gaussians_per_tile,
    flat_tile_id,
    flat_tile_u,
    flat_tile_v,
    local_tile_ids,
    local_tile_u,
    local_tile_v,
    starts_per_gaussian,
    umin,
    umax,
    vmin,
    vmax,
    umin_tile,
    umax_tile,
    vmin_tile,
    vmax_tile,
    n_u,
    n_v,
    num_tiles_per_gaussian,
)
if device.type == "mps":
    torch.mps.empty_cache()


## 9. Обратная ковариация и квадратичная форма

Для пикселя со смещением `(du, dv)` и обратной ковариацией

```text
A = [[A11, A12],
     [A12, A22]]
```

вычисляется:

```text
q = A11*du² + 2*A12*du*dv + A22*dv²
g = exp(-0.5*q)
alpha = opacity*g
```

Следующая ячейка строит обратные матрицы для всех on-screen
гауссианов и печатает настоящий пример.


In [ ]:
inverse_covariance = inv2x2(sigma_uv)
inverse_covariance[:, 0, 0] = torch.clamp(
    inverse_covariance[:, 0, 0],
    min=min_conis,
)
inverse_covariance[:, 1, 1] = torch.clamp(
    inverse_covariance[:, 1, 1],
    min=min_conis,
)

print("Sigma_uv[0] =\n", sigma_uv[0].detach().cpu())
print("inverse(Sigma_uv[0]) =\n", inverse_covariance[0].detach().cpu())
print("product =\n", (sigma_uv[0] @ inverse_covariance[0]).detach().cpu())


## 10. Растеризация одного настоящего тайла

`rasterize_tile` повторяет внутреннюю часть основного цикла:

1. восстанавливает `(tile_u, tile_v)` из линейного ID;
2. строит координаты `P` пикселей;
3. выбирает `G` гауссианов диапазона `[start, end)`;
4. формирует `q`, `g`, `alpha`, transmittance `T_i` и веса;
5. суммирует цвета.

Для диагностического тайла функция возвращает промежуточные тензоры.
Ячейка печатает первые реальные значения `q`, `alpha`, `T_i` и `w`, а
затем показывает получившийся tile-color.


In [ ]:
def rasterize_tile(group_index, return_debug=False):
    tile_id = int(unique_tile_ids[group_index].item())
    s0 = int(start[group_index].item())
    s1 = int(end[group_index].item())
    tile_gaussian_ids = gaussian_ids[s0:s1]

    tile_u = tile_id % num_tiles_u
    tile_v = tile_id // num_tiles_u
    x0 = tile_u * T
    y0 = tile_v * T
    x1 = min((tile_u + 1) * T, W)
    y1 = min((tile_v + 1) * T, H)

    xs = torch.arange(x0, x1, device=device, dtype=dtype)
    ys = torch.arange(y0, y1, device=device, dtype=dtype)
    pixel_u_grid, pixel_v_grid = torch.meshgrid(xs, ys, indexing="xy")
    pixel_u = pixel_u_grid.reshape(-1)
    pixel_v = pixel_v_grid.reshape(-1)
    pixel_indices = (pixel_v * W + pixel_u).to(torch.int64)

    gaussian_u = u[tile_gaussian_ids]
    gaussian_v = v[tile_gaussian_ids]
    gaussian_color = color[tile_gaussian_ids]
    gaussian_opacity = opacity[tile_gaussian_ids]
    gaussian_inverse_covariance = inverse_covariance[tile_gaussian_ids]

    du = pixel_u.unsqueeze(0) - gaussian_u.unsqueeze(-1)
    dv = pixel_v.unsqueeze(0) - gaussian_v.unsqueeze(-1)

    A11 = gaussian_inverse_covariance[:, 0, 0].unsqueeze(-1)
    A12 = gaussian_inverse_covariance[:, 0, 1].unsqueeze(-1)
    A22 = gaussian_inverse_covariance[:, 1, 1].unsqueeze(-1)
    q = A11 * du * du + 2 * A12 * du * dv + A22 * dv * dv

    inside = q <= chi_square_clip
    gaussian_value = torch.exp(-0.5 * torch.clamp(q, max=chi_square_clip))
    gaussian_value = torch.where(
        inside,
        gaussian_value,
        torch.zeros_like(gaussian_value),
    )

    alpha = (
        gaussian_opacity.unsqueeze(-1) * gaussian_value
    ).clamp_max(alpha_max)
    alpha = torch.where(
        alpha >= alpha_cutoff,
        alpha,
        torch.zeros_like(alpha),
    )

    transmittance = torch.cumprod(1 - alpha, dim=0)
    transmittance = torch.concatenate(
        [
            torch.ones((1, alpha.shape[-1]), device=device, dtype=dtype),
            transmittance[:-1],
        ],
        dim=0,
    )
    weights = alpha * transmittance
    tile_color = (
        weights.unsqueeze(-1) * gaussian_color.unsqueeze(1)
    ).sum(dim=0)

    debug = None
    if return_debug:
        debug = {
            "tile_id": tile_id,
            "tile_u": tile_u,
            "tile_v": tile_v,
            "bounds": (x0, y0, x1, y1),
            "gaussian_ids": tile_gaussian_ids,
            "pixel_u": pixel_u,
            "pixel_v": pixel_v,
            "q": q,
            "alpha": alpha,
            "transmittance": transmittance,
            "weights": weights,
        }

    return pixel_indices, tile_color, debug

sample_pixel_indices, sample_tile_color, debug = rasterize_tile(
    inspection_group_index,
    return_debug=True,
)

print("tile ID and coordinates:", debug["tile_id"], (debug["tile_u"], debug["tile_v"]))
print("pixel bounds (x0, y0, x1, y1):", debug["bounds"])
print("G Gaussians:", debug["q"].shape[0])
print("P pixels:", debug["q"].shape[1])
print("first pixel coordinate:", (float(debug["pixel_u"][0]), float(debug["pixel_v"][0])))
print("first 8 q values for the first pixel:\n", debug["q"][:8, 0].detach().cpu())
print("first 8 alpha values for the first pixel:\n", debug["alpha"][:8, 0].detach().cpu())
print("first 8 T_i values for the first pixel:\n", debug["transmittance"][:8, 0].detach().cpu())
print("first 8 weights for the first pixel:\n", debug["weights"][:8, 0].detach().cpu())

x0, y0, x1, y1 = debug["bounds"]
tile_height = y1 - y0
tile_width = x1 - x0
plt.figure(figsize=(3, 3))
plt.imshow(sample_tile_color.reshape(tile_height, tile_width, 3).detach().cpu().clamp(0, 1))
plt.title(f"Rendered tile {debug['tile_id']}")
plt.axis("off")
plt.show()

# The full debug tensors have shape [G, P]. Release them before the
# complete render so this inspection step does not increase peak memory.
del debug
if device.type == "mps":
    torch.mps.empty_cache()


## 11. Почему порядок гауссианов важен

Для гауссиана `i` накопленная прозрачность перед ним равна:

```text
T_i = product for j < i of (1 - alpha_j)
weight_i = alpha_i * T_i
```

Ближний гауссиан уменьшает вклад всех дальних. Поэтому одинаковый набор
гауссианов в другом порядке даст другой цвет. Стабильная сортировка из
шага 8 гарантирует front-to-back порядок внутри каждого тайла.

В предыдущей ячейке это можно увидеть на живых массивах: первая строка
`transmittance` равна единице, а следующие строки постепенно уменьшаются
после ненулевых alpha.


## 12. Полный проход по непустым тайлам

Теперь та же функция запускается для всех `U` непустых тайлов. Каждый
тайл пишет в собственный набор пикселей плоского буфера, поэтому записи
разных тайлов не пересекаются.

Эта ячейка соответствует основному циклу растеризатора из `3dgs.ipynb`.
На реальной сцене она является самой продолжительной частью ноутбука.


In [ ]:
final_image = torch.zeros((H * W, 3), device=device, dtype=dtype)

synchronize()
started_at = perf_counter()

for group_index in range(unique_tile_ids.shape[0]):
    pixel_indices, tile_color, _ = rasterize_tile(group_index)
    final_image[pixel_indices] = tile_color

synchronize()
elapsed = perf_counter() - started_at

rendered_image = final_image.reshape(H, W, 3).clamp(0, 1)

print(f"rendered {unique_tile_ids.shape[0]:,} non-empty tiles in {elapsed:.2f} s")
print("image shape:", tuple(rendered_image.shape))
print(
    "image min / mean / max:",
    float(rendered_image.min()),
    float(rendered_image.mean()),
    float(rendered_image.max()),
)


## 13. Итоговое изображение и фотография камеры

Слева показывается результат учебного 3DGS-растеризатора, справа —
исходная фотография выбранной COLMAP-камеры, уменьшенная до того же
разрешения.

Растеризатор использует только DC-составляющую цвета, поэтому точное
совпадение с фотографией не является целью этой ячейки.


In [ ]:
rendered_cpu = rendered_image.detach().cpu().numpy()
reference_image = Image.open(image_path).convert("RGB").resize(
    (W, H),
    Image.Resampling.LANCZOS,
)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
axes[0].imshow(rendered_cpu)
axes[0].set_title("Educational 3DGS rasterizer")
axes[1].imshow(reference_image)
axes[1].set_title("Reference camera image")
for axis in axes:
    axis.axis("off")
plt.tight_layout()
plt.show()


## 14. Финальная проверка инвариантов

Последняя ячейка не рендерит изображение заново. Она проверяет свойства,
от которых зависит корректность алгоритма:

- экранные ковариации конечны;
- ID тайлов отсортированы;
- внутри одного тайла сохранён глубинный порядок;
- диапазоны `[start, end)` полностью покрывают `K` пересечений;
- итоговое изображение конечно и лежит в `[0, 1]`.


In [ ]:
checks = {
    "finite inverse covariances": bool(torch.isfinite(inverse_covariance).all().item()),
    "sorted tile IDs": tile_order_ok,
    "depth order inside tiles": depth_order_ok,
    "groups cover all intersections": int(end[-1].item()) == num_tile_intersections,
    "finite final image": bool(torch.isfinite(rendered_image).all().item()),
    "final image in [0, 1]": bool(
        ((rendered_image >= 0) & (rendered_image <= 1)).all().item()
    ),
}

for name, passed in checks.items():
    print(f"{'PASS' if passed else 'FAIL'}: {name}")

assert all(checks.values())
print("\nAll rasterization invariants passed.")


## Сводка отличий от CUDA-референса

| Часть | CUDA-референс | Этот интерактивный ноутбук |
| --- | --- | --- |
| Устройство | CUDA | MPS с fallback на CUDA/CPU |
| Разложение 2×2 | `torch.linalg.eigh` | аналитический `eigh_2x2` |
| Пересечения тайлов | плотный `[Ns, max_u, max_v]` | компактный список длины `K` |
| Сортировка | составной `int64`-ключ | `stable=True` по `flat_tile_id` |
| Структура | одна большая функция | последовательные интерактивные стадии |
| Диагностика | итоговый рендер | формы и реальные значения на каждом шаге |

Попиксельное сравнение итоговых PNG после исправления сортировки показало
практически идентичный результат: отличались 33 пикселя из 1 619 801,
максимальная разница составляла 1 уровень RGB из 255.
